In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split

DATA_PATH = Path("../data/raw/ml-100k")

ratings = pd.read_csv(
    DATA_PATH / "u.data",
    sep="\t",
    names=["user_id", "movie_id", "rating", "timestamp"]
)

ratings.head()

,user_id,movie_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [2]:
train, test = train_test_split(
    ratings,
    test_size=0.2,
    random_state=42
)

print("Training ratings:", len(train))
print("Testing ratings:", len(test))

Training ratings: 80000
Testing ratings: 20000


In [3]:
global_mean = train["rating"].mean()

print(f"Global mean: {global_mean:.4f}")

Global mean: 3.5313


In [5]:
global_predictions = np.full(
    len(test),
    global_mean
)

global_rmse = np.sqrt(
    mean_squared_error(
        test["rating"],
        global_predictions
    )
)

global_mae = mean_absolute_error(
    test["rating"],
    global_predictions
)

print(f"Global Mean RMSE: {global_rmse:.4f}")
print(f"Global Mean MAE:  {global_mae:.4f}")

Global Mean RMSE: 1.1239
Global Mean MAE:  0.9420


In [8]:
movie_means = train.groupby("movie_id")["rating"].mean()

movie_predictions = test["movie_id"].map(movie_means)

movie_predictions = movie_predictions.fillna(global_mean)

movie_rmse = np.sqrt(
    mean_squared_error(
        test["rating"],
        movie_predictions
    )
)

movie_mae = mean_absolute_error(
    test["rating"],
    movie_predictions
)

print(f"Movie Mean RMSE: {movie_rmse:.4f}")
print(f"Movie Mean MAE:  {movie_mae:.4f}")

Movie Mean RMSE: 1.0210
Movie Mean MAE:  0.8123


In [10]:
user_means = train.groupby("user_id")["rating"].mean()

user_predictions = test["user_id"].map(user_means)

user_predictions = user_predictions.fillna(global_mean)

user_rmse = np.sqrt(
    mean_squared_error(
        test["rating"],
        user_predictions
    )
)

user_mae = mean_absolute_error(
    test["rating"],
    user_predictions
)

print(f"User Mean RMSE: {user_rmse:.4f}")
print(f"User Mean MAE:  {user_mae:.4f}")


User Mean RMSE: 1.0417
User Mean MAE:  0.8346


In [11]:
baseline_results = pd.DataFrame({
    "Model": [
        "Global Mean",
        "Movie Mean",
        "User Mean"
    ],
    "RMSE": [
        global_rmse,
        movie_rmse,
        user_rmse
    ],
    "MAE": [
        global_mae,
        movie_mae,
        user_mae
    ]
})

baseline_results.sort_values("RMSE")

,Model,RMSE,MAE
1,Movie Mean,1.020964,0.812273
2,User Mean,1.041745,0.834636
0,Global Mean,1.123860,0.941955


## Baseline Results

The Movie Mean baseline achieved the best performance with an RMSE of approximately 1.021 and an MAE of 0.812. This outperformed both the User Mean and Global Mean baselines.

The improvement over the Global Mean model demonstrates that accounting for differences between individual movies provides useful predictive information. User-specific rating tendencies also improve performance, although the Movie Mean model performed slightly better.

These models provide benchmarks for evaluating more sophisticated recommendation approaches. Future models should ideally achieve an RMSE below 1.021 while also providing personalized recommendations.